In [254]:
from sklearn.datasets import load_iris

from sklearn.model_selection import train_test_split

from sklearn.naive_bayes import GaussianNB

In [255]:
from collections import Counter
from nltk.util import ngrams

import numpy as np

In [256]:
from pathlib import Path
import numpy as np
import pickle

# Get verses from text-fabric

In [257]:
# prepare Peshitta OT from text-fabric
from tf.app import use

In [258]:
def get_verses(
    target_books: dict[str, list[int]],
    target_fabric: str="etcbc/peshitta",
    ver: str="0.2") -> dict[str, list[tuple[str, list[str]]]]:
    
    api_handler = use(target_fabric, hoist=globals(), version=ver)
    result_verses = None
    
    for book in Fs("book@en").items():    
        if book[1] in target_books.keys():
            book_verses = []
            print(book[1])
            chapters = L.d(book[0], otype="chapter")
            # results = "Verse Reference,Lemmatised Transliteration,Plain Transliteration,Original Syriac Text\n"
            num_book_verses = 0
            for chapter in chapters:
                if int(F.chapter.v(chapter)) in target_books[book[1]]:
                    for verse in L.d(chapter, otype="verse"):
                        verse_ref = f"{book[1]} Chapter {F.chapter.v(chapter)} Verse {F.verse.v(verse)}"
                        # get all words in this verse
                        words = L.d(verse, otype="word")
                        # transliteration of this verse as a list of words
                        translit_verse = [F.word_etcbc.v(w_id) for w_id in words]
                        book_verses.append((verse_ref, translit_verse))
            if result_verses is None:
                result_verses = {book[1]: book_verses}
            else:
                result_verses.update({book[1]: book_verses})
    return result_verses

In [259]:
# Try Fs("book@en").items() to get names of the book included in the text-fabric dataset
ot_train_books = {
    "Genesis": list(range(1,51)),
    "Exodus": list(range(1,22))
}

ot_test_books = {
    "Deuteronomy": list(range(1,21))
}

In [260]:
# Parse the dataset and get verses
# each verse in a tuple "(`verse reference`: str, `transliteration as a list of words`: list[str])"

ot_train_book_verses = get_verses(ot_train_books)
print(ot_train_book_verses)

ot_test_book_verses = get_verses(ot_test_books)

**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
book,65,6566.69,100
chapter,1269,336.36,100
verse,31341,13.62,100
word,426835,1.00,100


Genesis
Exodus
{'Genesis': [('Genesis Chapter 1 Verse 1', ['BRCJT', 'BR>', '>LH>', 'JT', 'CMJ>', 'WJT', '>R<>']), ('Genesis Chapter 1 Verse 2', ['>R<>', 'HWT', 'TWH', 'WBWH^', 'WXCWK>', '<L', '>"PJ', 'THWM>', 'WRWXH', 'D>LH>', 'MRXP>', '<L', '>"PJ', 'M"J>']), ('Genesis Chapter 1 Verse 3', ['W>M#R', '>LH>', 'NHW>', 'NWHR>', 'WHW#>', 'NWHR>']), ('Genesis Chapter 1 Verse 4', ['WX#Z>', '>LH>', 'LNWHR>', 'DCPJR', 'WP#RC', '>LH>', 'BJT', 'NWHR>', 'LXCWK>']), ('Genesis Chapter 1 Verse 5', ['WQ#R>', '>LH>', 'LNWHR>', '>JMM>', 'WLXCWK>', 'Q#R>', 'LLJ>', 'WHW#>', 'RMC>', 'WHW>', 'YPR>', 'JWM>', 'XD']), ('Genesis Chapter 1 Verse 6', ['W>M#R', '>LH>', 'NHW>', '>RQJ<>', 'BMY<T', 'M"J>', 'WNHW>', 'PRC^', 'BJT', 'M"J>', 'LM"J>']), ('Genesis Chapter 1 Verse 7', ['W<B#D', '>LH>', '>RQJ<>', 'WP#RC', 'BJT', 'M"J>', 'DLTXT', 'MN', '>RQJ<>', 'WBJT', 'M"J>', 'DL<L', 'MN', '>RQJ<>', 'WHW#>', 'HKN>']), ('Genesis Chapter 1 Verse 8', ['WQ#R>', '>LH>', 'L>RQJ<>', 'CMJ>', 'WHW#>', 'RMC>', 'WHW>', 'YPR>', 'JWM>', 

**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
book,65,6566.69,100
chapter,1269,336.36,100
verse,31341,13.62,100
word,426835,1.00,100


Deuteronomy


In [261]:
nt_train_books = {
    "Matthew": list(range(1,31)),
    "Mark": list(range(1,17)),
    "Luke": list(range(1,24)),
    "John": list(range(1,22)),
}

nt_test_books = {
    "Acts": list(range(1,29))
}

In [262]:
nt_train_book_verses = get_verses(nt_train_books, target_fabric="etcbc/syrnt", ver="0.1")
# print(nt_train_book_verses)

nt_test_book_verses = get_verses(nt_test_books, target_fabric="etcbc/syrnt", ver="0.1")

**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
book,27,4060.74,100
chapter,260,421.69,100
lexeme,3038,36.09,100
verse,7957,13.78,100
word,109640,1.00,100


Matthew
Mark
Luke
John


**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
book,27,4060.74,100
chapter,260,421.69,100
lexeme,3038,36.09,100
verse,7957,13.78,100
word,109640,1.00,100


Acts


## Create a list of all verses for training

In [263]:
train_data_dict = ot_train_book_verses.copy()
train_data_dict.update(nt_train_book_verses)

ot_train_verses = [vrs for verses in ot_train_book_verses.values() for vrs in verses]
nt_train_verses = [vrs for verses in nt_train_book_verses.values() for vrs in verses]

print(len(ot_train_verses))
print(len(nt_train_verses))

train_verse_txts = ot_train_verses.copy()
train_verse_txts.extend(nt_train_verses)

train_verse_labels = [0 for i in range(len(ot_train_verses))]
train_verse_labels.extend([1 for i in range(len(nt_train_verses))])
print(len(train_verse_labels))
print(len(train_verses))

2115
3726
5841
5841


## Prepare the texts and labels

In [264]:
train_verse_txts = []
train_verse_labels = []

for verse in train_verses:
    train_verse_txts.append(verse[0])
    train_verse_labels.append(verse[1])

# Generate Bag-of-Words counters

In [265]:
from collections.abc import Callable

In [266]:
def count_n_grams(
        word_count: int,
        verse_words: list[str],
        ngram_formatter: Callable,
        span: int=3
    ) -> (int, Counter):
    
    # Format the verse to feed into ngrams()
    formatted_inputs = ngram_formatter(verse_words)
    
    # Generate and count ngrams
    n_grams = list(ngrams(formatted_inputs, span))
    n_gram_counts = Counter(n_grams)
    
    # Keep the ngram counters
    return (word_count+len(verse_words), n_gram_counts)

In [267]:
def make_vocab(verses: list[tuple[str, list[str]]], ngram_formatter: Callable, span: int=3):
    word_cnt = 0
    n_gram_vocabs = None
    n_gram_counters = []

    for verse in verses:
        word_cnt, n_gram_counter = count_n_grams(word_cnt, verse[1], ngram_formatter, span)

        n_gram_counters.append(n_gram_counter)
    
        if n_gram_vocabs is None:
            n_gram_vocabs = n_gram_counter.copy()
        else:
            n_gram_vocabs.update(n_gram_counter)

    print(word_cnt)
    return (n_gram_vocabs, n_gram_counters)

In [268]:
def identity(input):
    return input

def make_word_n_gram_vocab(verses: list[tuple[str, list[str]]], span: int=3):
    return make_vocab(verses, indentity, span)

def make_char_n_gram_vocab(verses: list[tuple[str, list[str]]], span: int=3):
    return make_vocab(verses, ' '.join, span)

In [269]:
char_n_gram_vocabs = make_char_n_gram_vocab(train_verse_txts)
print(f"Found {len(char_n_gram_vocabs[0])} independent n-grams from {len(char_n_gram_vocabs[1])} verses!")

78289
Found 7441 independent n-grams from 5841 verses!


In [270]:
print(char_n_gram_vocabs[0])
# print(len(char_n_gram_vocabs.keys()))

Counter({('W', 'N', ' '): 4922, ('J', 'N', ' '): 4797, ('>', ' ', 'D'): 4133, ('N', '>', ' '): 3406, ('>', ' ', 'W'): 3355, ('L', '>', ' '): 2895, ('T', '>', ' '): 2850, (' ', '>', 'N'): 2700, ('>', ' ', '>'): 2690, ('>', ' ', 'L'): 2483, (' ', 'H', 'W'): 2378, (' ', 'W', '>'): 2289, (' ', 'L', 'H'): 2242, ('M', 'N', ' '): 2202, (' ', 'M', 'N'): 2188, ('J', '>', ' '): 2128, ('>', 'M', 'R'): 2083, ('N', ' ', '>'): 1942, ('R', '>', ' '): 1902, (' ', 'D', '>'): 1788, ('>', ' ', 'M'): 1764, ('H', 'W', 'N'): 1731, ('H', 'J', ' '): 1669, (' ', 'L', '>'): 1659, (' ', 'D', 'J'): 1656, ('N', ' ', 'D'): 1626, ('M', '>', ' '): 1602, ('>', ' ', 'H'): 1601, ('L', 'H', ' '): 1553, ('D', 'J', 'N'): 1541, ('W', 'H', 'J'): 1539, ('W', '>', 'M'): 1444, ('N', ' ', 'L'): 1435, ('M', 'R', ' '): 1416, ('N', ' ', 'W'): 1399, ('W', '>', ' '): 1321, (' ', '>', 'J'): 1319, ('>', ' ', 'B'): 1314, ('H', 'W', '>'): 1251, ('R', ' ', 'L'): 1244, ('>', 'N', 'T'): 1199, ('J', 'T', ' '): 1180, ('C', '>', ' '): 1174, ('

# Vectorise the verses

In [271]:
# Count the number of each n-gram per verse,
# reusing the counters created while constructing the vocabulary

assert(len(char_n_gram_vocabs[1]) == len(train_verse_txts))

In [272]:
def make_bow(vocabulary: dict):
    return dict.fromkeys(vocabulary.keys(), 0)

In [273]:
def make_feature(n_gram_vocabs: tuple[Counter, list[Counter]]):
    feature_list = []
    for i in range(len(n_gram_vocabs[1])):
        # Make a skeleton dict to use as the BoW feature, set all counts to 0
        n_gram_bow = make_bow(n_gram_vocabs[0])
        # Add counts to words that ecist in this verse
        n_gram_bow.update(n_gram_vocabs[1][i])
        verse_features = list(n_gram_bow.values())
        feature_list.append(verse_features)
    return feature_list

In [274]:
char_n_gram_feat = make_feature(char_n_gram_vocabs)

In [275]:
print(len(char_n_gram_feat))

5841


# Train classifiers

In [276]:
# n_gram_train, n_gram_test, n_gram_train_y, n_gram_test_y = train_test_split(n_gram_feats, labels, test_size=0.2, random_state=0)

gnb = GaussianNB()

c_clf = gnb.fit(char_n_gram_feat, train_verse_labels)

print(f"{train_verse_labels[0]}, {train_verse_labels[-1]}")

0, 1


## Prepare test data

In [277]:
def merge_counts(counter: Counter, feat_vec: dict):
    for n_gram in counter.keys():
        # print(n_gram)
        if n_gram in feat_vec.keys():
            feat_vec[n_gram] = counter[n_gram]

In [278]:
def vectorise(
        verses: list[tuple[str, list[str]]],
        bag_of_words: Counter,
        ngram_formatter: Callable,
        span: int=3
    ) -> list[list[int]]:
    # Get the verse of format [(VERSE_REF, [VERSE_WORDS])]
    # Generate two lists: [[N_GRAM_COUNTS] per each verse] & [LABEL per each verse]
    labels = []
    wc = 0
    verse_n_grams = []
    for verse in verses:
        wc, local_n_gram_counts = count_n_grams(word_count=wc, verse_words=verse[1], ngram_formatter=ngram_formatter, span=3)
        n_gram_bow = dict.fromkeys(bag_of_words.keys(), 0)
        merge_counts(local_n_gram_counts, n_gram_bow)
        verse_n_grams.append(list(n_gram_bow.values()))
    print(f"Parsed {wc} words from {len(verses)} verses")
    return verse_n_grams

### OT

In [279]:
ot_test_verses = [vrs for book in ot_test_book_verses.values() for vrs in book]
test_y_np = np.empty(len(ot_test_verses), dtype=int)
test_y_np.fill(0)
test_X = vectorise(ot_test_verses, make_bow(char_n_gram_vocabs[0]), ' '.join)
print(test_y_np.shape)

Parsed 8348 words from 555 verses
(555,)


In [280]:
test_X_np = np.array(test_X)

y_pred = c_clf.predict(test_X_np)

num_neg_samples = test_X_np.shape[0]
num_fp = ((test_y_np != y_pred).sum())
num_tn = num_neg_samples - num_fp

print("Number of mislabeled points out of a total %d OT verses: %d" % (num_neg_samples, num_fp))
print(f"Local accuracy: {(num_tn/num_neg_samples):.02f}")

Number of mislabeled points out of a total 555 OT verses: 76
Local accuracy: 0.86


### NT

In [290]:
# Labels
nt_test_verses = [vrs for book in nt_test_book_verses.values() for vrs in book]
nt_test_y_np = np.empty(len(nt_test_verses), dtype=int)
nt_test_y_np.fill(1)
print(nt_test_y_np.shape)

# Features
nt_test_X = vectorise(nt_test_verses, make_bow(char_n_gram_vocabs[0]), ' '.join)
nt_test_X_np = np.array(nt_test_X)

(1007,)
Parsed 15383 words from 1007 verses


In [292]:
nt_y_pred = c_clf.predict(nt_test_X_np)

num_pos_samples = nt_test_X_np.shape[0]
num_fn = ((nt_test_y_np != nt_y_pred).sum())
num_tp = num_pos_samples - num_fn

print("Number of mislabeled points out of a total %d NT verses: %d" % (num_pos_samples, num_fn))
print(f"Local accuracy: {(num_tp/num_pos_samples):.02f}")

Number of mislabeled points out of a total 1007 NT verses: 57
Local accuracy: 0.94


In [293]:
precision = num_tp / (num_tp+num_fp)
recall = num_tp / (num_tp+num_fn)
f1_score = 2 * (precision * recall) / (precision + recall)

print(f"Overall Precision: {precision:.02f}")
print(f"Overall Recall: {recall:.02f}")
print(f"F1 Score: {f1_score:.02f}")

Overall Precision: 0.93
Overall Recall: 0.94
F1 Score: 0.93


In [ ]:
# pos = sum([book[1] for book in ot_verse_nums])
# print(pos)
pos = 0

ot_mislabels = {
    "Deuteronomy": []
}

for book in ot_verse_nums:
    print(book[0])
    book_verses = []
    verse_labels = []
    for i in range(book[1]):
        book_verses.append(verse_chargram_feat[pos])
        verse_labels.append(0)
        pos += 1

    book_verses = np.array(book_verses)
    verse_labels = np.array(verse_labels)
    book_pred = c_clf.predict(book_verses)
    if book[0] == "Genesis":
        print(book_pred)
    mislabeled_points = [i for i in range(len(book_pred)) if book_pred[i] == 1]
    print("Number of mislabeled points out of a total %d points : %d" % (book_verses.shape[0], (verse_labels != book_pred).sum()))

with Path(f"./out/prediction_data_{book[0]}.json").open(mode="w") as fp:
    json.dump(mislabels, fp)